# 01 — CIFAR-10 data exploration

Quick look at the dataset before we start optimizing. The point here is **not** classification, but understanding what the optimizers will be working on: image shape, per-class distribution, pixel statistics, and how the flattened representation looks for the convex baselines.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np
import matplotlib.pyplot as plt
from src.data_loader import load_cifar10_numpy, CLASS_NAMES

In [ ]:
X_train, y_train, X_test, y_test = load_cifar10_numpy(flatten=False, normalize=False)
print('train:', X_train.shape, y_train.shape)
print('test :', X_test.shape, y_test.shape)
print('pixel range:', X_train.min(), X_train.max())

In [ ]:
# Class balance
counts = np.bincount(y_train)
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(10), counts)
ax.set_xticks(range(10))
ax.set_xticklabels(CLASS_NAMES, rotation=45)
ax.set_ylabel('train samples')
ax.set_title('CIFAR-10 — class balance')
plt.tight_layout()

In [ ]:
# Sample grid
fig, axes = plt.subplots(10, 8, figsize=(10, 12))
for c in range(10):
    idx = np.where(y_train == c)[0][:8]
    for j, k in enumerate(idx):
        axes[c, j].imshow(X_train[k].transpose(1, 2, 0))
        axes[c, j].axis('off')
    axes[c, 0].set_title(CLASS_NAMES[c], loc='left', fontsize=10)
plt.tight_layout()

## Pixel statistics

For the convex baselines we feed flattened images. Per-channel mean/std are useful to understand what the inputs of the logistic/SVM model look like, and why pre-normalization helps the conditioning (and therefore the convergence rate) of first-order methods.

In [ ]:
for c, name in enumerate(['R', 'G', 'B']):
    print(f'{name}: mean={X_train[:, c].mean():.3f}  std={X_train[:, c].std():.3f}')

**Discussion (to include in the report):** the input matrix for a linear convex model on raw pixels is fairly ill-conditioned — neighboring pixels are strongly correlated, which inflates the condition number of the Hessian and explains why plain gradient descent converges slowly even on a strictly convex problem. This is one of the empirical hooks for the report.